# score_runs_sp_val_i — canonical scorer for the soft-prompt val slices
The scorer from `score_runs.ipynb`, unchanged: same break patch, same `resize_ref`, same `score_row`, same `max_workers=8`, scored against the original `test.jsonl`-style instruction and never an augmented one. The only wrapping is an outer loop over steps 500/1000/2000/3000, writing `val__{RUN}_step{N}.jsonl` per step.

Set `VARIANT` per notebook and run three copies in parallel. Rate limits crash a pass non-destructively — rows flush per row, the done-set skips them, error rows stay retryable — so re-run the scoring cell until each step prints `to score 0`.

## 1. Repo, scorer patch

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
assert _head == COMMIT, f"repo is at {_head[:7]}, expected {COMMIT[:7]}"
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [2]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "val"                       # 98-row held-out slice for step selection
VARIANT = "i"                         # i | ii | iii — set per notebook
RUN     = {"i": "sp_i_p668", "ii": "sp_ii_p668", "iii": "sp_iii_p139"}[VARIANT]
STEPS   = [500, 1000, 2000, 3000]

for step in STEPS:
    gen = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(gen) if f.endswith(".png")]) if os.path.isdir(gen) else 0
    print(f"{RUN} step {step}: {n} images at {gen}")

Mounted at /content/drive
sp_i_p668 step 500: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/sp_i_p668__step500
sp_i_p668 step 1000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/sp_i_p668__step1000
sp_i_p668 step 2000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/sp_i_p668__step2000
sp_i_p668 step 3000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/sp_i_p668__step3000


## 3. Reference images

In [3]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
all_have = sorted({int(f[:-4])
                   for step in STEPS
                   for f in os.listdir(f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}")
                   if os.path.isdir(f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}") and f.endswith(".png") and f[:-4].isdigit()})
assert all_have, f"no generated images found for {RUN} across steps {STEPS}"
assert max(all_have) < len(rows), f"image index {max(all_have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in all_have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | {len(all_have)} generated indices across {len(STEPS)} steps")

restored 37063 / 37063
val: 98 rows | 98 generated indices across 4 steps


## 4. Sanity — 3 rows

In [4]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

_smoke_step = STEPS[0]
_smoke_GEN  = f"{DRIVE}/gen/{DATASET}/{RUN}__step{_smoke_step}"
_smoke_have = sorted(int(f[:-4]) for f in os.listdir(_smoke_GEN) if f.endswith(".png") and f[:-4].isdigit())
for i in _smoke_have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{_smoke_GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"step {_smoke_step} row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

step 500 row 0: PF=6  SC=2
   PF: The second image shows a professional man in a dark suit with a crisp white shirt and patterned tie, holding a sleek laptop. He is smiling a
   SC: The identity of the man in the final image (second image) does not match the subject in the reference image (first image); the facial featur

step 500 row 1: PF=8  SC=9
   PF: The image accurately depicts a professional man in a dark, tailored suit, complete with a crisp white shirt and patterned tie, standing conf
   SC: The laptop in the final image is almost perfectly consistent with the one in the reference image, matching in shape, bezel, keyboard layout,

step 500 row 2: PF=7  SC=6
   PF: The final image accurately features a professional man standing confidently and smiling, dressed in a tailored suit with a white shirt and p
   SC: The final image mostly captures the overall professional theme, but there are several inconsistencies regarding identity. The man's facial f



## 5. Full pass per checkpoint (resumable)

In [5]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

# identical scorer to score_runs.ipynb; only wrapped to run each step's directory in turn.
# rate limits crash a pass non-destructively: rows are flushed per row, the done-set skips them,
# error rows stay retryable -> just re-run this cell until every step prints "to score 0".
for step in STEPS:
    GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl"
    have   = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())

    done = set()
    if os.path.exists(SCORES):
        for l in open(SCORES, encoding="utf-8"):
            try:
                d = json.loads(l)
                if "error" not in d: done.add(d["i"])   # failures stay retryable
            except: pass

    def score_row(i):
        r  = rows[i]
        gp = f"{GEN}/{i:05d}.png"
        if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
        refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
        gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
        time.sleep(0.2)
        res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
        pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
        if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
        # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
        if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
            return {"i": i, "error": "guessed_score"}
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
                "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                "PF": pf, "SC": sc,
                "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

    todo = [i for i in have if i not in done]
    print(f"\nstep {step}: to score {len(todo)} (done: {len(done)}) -> {SCORES}")
    with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(score_row, i) for i in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
            if n % 25 == 0: print(f"  {n}/{len(todo)}")
    print(f"step {step} pass done -> {SCORES}")
print("\nall steps: re-run this cell until each prints 'to score 0'")


step 500: to score 98 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step500.jsonl
  25/98
  50/98
  75/98
step 500 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step500.jsonl

step 1000: to score 98 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step1000.jsonl
  25/98
  50/98
  75/98
step 1000 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step1000.jsonl

step 2000: to score 98 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step2000.jsonl
  25/98
  50/98
  75/98
step 2000 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step2000.jsonl

step 3000: to score 98 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__sp_i_p668_step3000.jsonl
  25/98
  50/98

## 6. Step-selection curve

In [6]:
import math, json
import numpy as np
ov = lambda r: math.sqrt(max(r["PF"],0)*max(r["SC"],0))

# selection is argmax val Overall, pre-registered. SC is printed alongside because it is the
# component the arm is expected to move, not because it selects.
print(f"{RUN}: step-selection curve on the {DATASET} slice\n")
print(f"{'step':>6s} {'n':>4s} {'err':>4s} {'PF':>7s} {'SC':>7s} {'Overall':>8s}")
curve = {}
for step in STEPS:
    p = f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl"
    raw = [json.loads(l) for l in open(p, encoding="utf-8")]
    rs  = [r for r in raw if "error" not in r]
    seen = {}
    for r in rs: seen[r["i"]] = r          # last write wins if a step was scored twice
    rs = list(seen.values())
    curve[step] = (np.mean([r["PF"] for r in rs]), np.mean([r["SC"] for r in rs]), np.mean([ov(r) for r in rs]))
    print(f"{step:>6d} {len(rs):>4d} {len(raw)-len(rs):>4d} {curve[step][0]:>7.3f} {curve[step][1]:>7.3f} {curve[step][2]:>8.3f}")

sel = max(STEPS, key=lambda s: curve[s][2])
print(f"\nselected checkpoint: {sel}  (val Overall {curve[sel][2]:.3f})")
print(f"argmax SC for reference: {max(STEPS, key=lambda s: curve[s][1])}")
print("checkpoint-to-checkpoint sigma_d on this slice was 0.973 for the LoRA arm, so the smallest")
print(f"difference resolvable at n={len(rows)} is about 0.28 Overall; treat closer gaps as ties.")
print("confirm the selected step against baseline on the full 488 test set before reporting.")

sp_i_p668: step-selection curve on the val slice

  step    n  err      PF      SC  Overall
   500   98    0   6.551   6.347    6.311
  1000   98    0   6.898   6.449    6.566
  2000   98    0   7.020   6.633    6.661
  3000   98    0   6.878   6.480    6.561

selected checkpoint: 2000  (val Overall 6.661)
argmax SC for reference: 2000
checkpoint-to-checkpoint sigma_d on this slice was 0.973 for the LoRA arm, so the smallest
difference resolvable at n=98 is about 0.28 Overall; treat closer gaps as ties.
confirm the selected step against baseline on the full 488 test set before reporting.
